# Proyecto - Predicción de Importaciones para Cencosud

Este notebook implementa un flujo de trabajo reproducible para estimar las importaciones de electrodomésticos en Chile con foco en apoyar la planificación logística y de abastecimiento de **Cencosud**.

## Objetivo
Construir un modelo predictivo que permita estimar el comportamiento futuro de las importaciones de electrodomésticos en Chile.

## Variables analizadas
- `valor_cif`
- `peso`
- `cantidad`

## Códigos HS considerados
- `8418`: Refrigeradores
- `8450`: Lavadoras
- `8516`: Microondas / hornos eléctricos
- `8528`: Televisores

## Horizonte de predicción
- `6 meses`

## Modelos a comparar
- ARIMA
- Regresión Lineal
- Random Forest
- XGBoost
- LightGBM

> **Nota importante**: Este notebook fue corregido para trabajar de forma coherente con columnas en **minúscula** y con los archivos:
> - `importaciones_hs_filtrado_estandarizado.csv`
> - `importaciones_hs_filtrado_raw.csv`

## BLOQUE 12 — EDA general

In [ ]:
if "monthly_total" in globals():
    print("Resumen estadístico mensual total:")
    display(monthly_total.describe())

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_total["fecha_mes"], monthly_total["valor_cif"])
    plt.title("Serie mensual total - Valor CIF")
    plt.xlabel("Fecha")
    plt.ylabel("Valor CIF")
    plt.xticks(rotation=45)
    plt.tight_layout()
    guardar_figura("serie_mensual_valor_cif_total.png")
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_total["fecha_mes"], monthly_total["peso"])
    plt.title("Serie mensual total - Peso")
    plt.xlabel("Fecha")
    plt.ylabel("Peso")
    plt.xticks(rotation=45)
    plt.tight_layout()
    guardar_figura("serie_mensual_peso_total.png")
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_total["fecha_mes"], monthly_total["cantidad"])
    plt.title("Serie mensual total - Cantidad")
    plt.xlabel("Fecha")
    plt.ylabel("Cantidad")
    plt.xticks(rotation=45)
    plt.tight_layout()
    guardar_figura("serie_mensual_cantidad_total.png")
    plt.show()
else:
    print("[WARNING] No existe monthly_total.")

## BLOQUE 13 — EDA por HS

In [ ]:
if "monthly_by_hs" in globals():
    for variable in VARIABLES_OBJETIVO:
        plt.figure(figsize=(12, 6))
        for hs in sorted(monthly_by_hs["hs4"].unique()):
            temp = monthly_by_hs[monthly_by_hs["hs4"] == hs]
            plt.plot(temp["fecha_mes"], temp[variable], label=hs)

        plt.title(f"Serie mensual por HS - {variable}")
        plt.xlabel("Fecha")
        plt.ylabel(variable)
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        guardar_figura(f"serie_por_hs_{variable}.png")
        plt.show()
else:
    print("[WARNING] No existe monthly_by_hs.")

## BLOQUE 14 — Modelado: preparación de datos

In [ ]:
resultados_metricas = []
resultados_predicciones = {}
resultados_forecast = {}

if "monthly_total" in globals():
    print("Iniciando modelado...")
else:
    print("[ERROR] No existe monthly_total; no se puede modelar.")

## BLOQUE 15 — Modelado por variable objetivo

In [ ]:
if "monthly_total" in globals():
    for target_col in VARIABLES_OBJETIVO:
        print("=" * 80)
        print(f"VARIABLE OBJETIVO: {target_col}")
        print("=" * 80)

        df_feat = crear_features_temporales(monthly_total, target_col)

        # 1) ARIMA
        mets, pred_df, err = entrenar_y_evaluar_arima(monthly_total, target_col)
        if mets is not None:
            resultados_metricas.append({"variable": target_col, **mets})
            resultados_predicciones[f"{target_col}_ARIMA"] = pred_df
            forecast_df = forecast_arima(monthly_total, target_col)
            if forecast_df is not None:
                forecast_df["modelo"] = "ARIMA"
                forecast_df["variable"] = target_col
                resultados_forecast[f"{target_col}_ARIMA"] = forecast_df
            print("[OK] ARIMA evaluado.")
        else:
            print(f"[WARNING] ARIMA no disponible para {target_col}: {err}")

        # 2) Regresión Lineal
        mets, pred_df, err = entrenar_y_evaluar_supervisado(
            df_feat, target_col, "Regresión Lineal", LinearRegression()
        )
        if mets is not None:
            resultados_metricas.append({"variable": target_col, **mets})
            resultados_predicciones[f"{target_col}_RegresionLineal"] = pred_df
            forecast_df = forecast_supervisado(df_feat, target_col, LinearRegression())
            if forecast_df is not None:
                forecast_df["modelo"] = "Regresión Lineal"
                forecast_df["variable"] = target_col
                resultados_forecast[f"{target_col}_RegresionLineal"] = forecast_df
            print("[OK] Regresión Lineal evaluada.")
        else:
            print(f"[WARNING] Regresión Lineal no disponible para {target_col}: {err}")

        # 3) Random Forest
        mets, pred_df, err = entrenar_y_evaluar_supervisado(
            df_feat, target_col, "Random Forest",
            RandomForestRegressor(n_estimators=200, random_state=42, max_depth=6)
        )
        if mets is not None:
            resultados_metricas.append({"variable": target_col, **mets})
            resultados_predicciones[f"{target_col}_RandomForest"] = pred_df
            forecast_df = forecast_supervisado(
                df_feat, target_col,
                RandomForestRegressor(n_estimators=200, random_state=42, max_depth=6)
            )
            if forecast_df is not None:
                forecast_df["modelo"] = "Random Forest"
                forecast_df["variable"] = target_col
                resultados_forecast[f"{target_col}_RandomForest"] = forecast_df
            print("[OK] Random Forest evaluado.")
        else:
            print(f"[WARNING] Random Forest no disponible para {target_col}: {err}")

        # 4) XGBoost
        if xgboost_disponible:
            mets, pred_df, err = entrenar_y_evaluar_supervisado(
                df_feat, target_col, "XGBoost",
                XGBRegressor(
                    n_estimators=200,
                    learning_rate=0.05,
                    max_depth=4,
                    random_state=42
                )
            )
            if mets is not None:
                resultados_metricas.append({"variable": target_col, **mets})
                resultados_predicciones[f"{target_col}_XGBoost"] = pred_df
                forecast_df = forecast_supervisado(
                    df_feat, target_col,
                    XGBRegressor(
                        n_estimators=200,
                        learning_rate=0.05,
                        max_depth=4,
                        random_state=42
                    )
                )
                if forecast_df is not None:
                    forecast_df["modelo"] = "XGBoost"
                    forecast_df["variable"] = target_col
                    resultados_forecast[f"{target_col}_XGBoost"] = forecast_df
                print("[OK] XGBoost evaluado.")
            else:
                print(f"[WARNING] XGBoost no disponible para {target_col}: {err}")
        else:
            print("[INFO] XGBoost no instalado.")

        # 5) LightGBM
        if lightgbm_disponible:
            mets, pred_df, err = entrenar_y_evaluar_supervisado(
                df_feat, target_col, "LightGBM",
                LGBMRegressor(
                    n_estimators=200,
                    learning_rate=0.05,
                    max_depth=4,
                    random_state=42
                )
            )
            if mets is not None:
                resultados_metricas.append({"variable": target_col, **mets})
                resultados_predicciones[f"{target_col}_LightGBM"] = pred_df
                forecast_df = forecast_supervisado(
                    df_feat, target_col,
                    LGBMRegressor(
                        n_estimators=200,
                        learning_rate=0.05,
                        max_depth=4,
                        random_state=42
                    )
                )
                if forecast_df is not None:
                    forecast_df["modelo"] = "LightGBM"
                    forecast_df["variable"] = target_col
                    resultados_forecast[f"{target_col}_LightGBM"] = forecast_df
                print("[OK] LightGBM evaluado.")
            else:
                print(f"[WARNING] LightGBM no disponible para {target_col}: {err}")
        else:
            print("[INFO] LightGBM no instalado.")

## BLOQUE 16 — Tabla comparativa de métricas

In [ ]:
if len(resultados_metricas) > 0:
    metricas_df = pd.DataFrame(resultados_metricas)
    metricas_df = metricas_df.sort_values(["variable", "rmse", "mae"]).reset_index(drop=True)

    print("Métricas comparativas:")
    display(metricas_df)

    guardar_tabla(metricas_df, "metricas_comparativas_modelos.csv")
else:
    print("[WARNING] No se generaron métricas.")

## BLOQUE 17 — Mejor modelo por variable

In [ ]:
if "metricas_df" in globals():
    mejores_modelos = (
        metricas_df.sort_values(["variable", "rmse", "mae"])
        .groupby("variable", as_index=False)
        .first()
    )

    print("Mejor modelo por variable:")
    display(mejores_modelos)

    guardar_tabla(mejores_modelos, "mejores_modelos_por_variable.csv")
else:
    print("[WARNING] No existe metricas_df.")